# PTA Array NUTS Sampling

Bayesian inference over a full pulsar timing array with multiple continuous
gravitational wave (CW) sources using BlackJax NUTS.

**Joint parameter space:**
- Per-pulsar timing parameters (RAJ, DECJ, F0, F1, DM, PX) — whitened via WLS Cholesky
- Per-CW-source parameters (log10_h, cos_gwtheta, gwphi, log10_fgw, cos_inc, psi, phase0) — logit-transformed to unconstrained space

**Workflow:**
1. Generate N synthetic pulsars with random sky positions and timing parameters
2. Inject M CW signals into the TOAs
3. WLS-fit each pulsar individually for starting points and covariance estimates
4. Build the joint log-probability in a transformed (unconstrained) coordinate system
5. Run NUTS warmup + sampling via BlackJax
6. Diagnostics: CW parameter recovery, trace plots, corner plots

In [1]:
# ---- User Configuration ----
N_PULSARS = 5
M_CW_SOURCES = 2
N_TOAS = 200
START_MJD = 57000.0
END_MJD = 60000.0       # ~8 yr observation span
TOA_ERROR = 1e-6         # 1 μs white noise (realistic for PTA pulsars)
FREQ = 1400.0            # MHz
SEED = 42

# NUTS configuration
N_WARMUP = 1000
N_SAMPLES = 2000

# CW prior bounds (for logit transform)
CW_BOUNDS = {
    "log10_h": (-18.0, -10.0),
    "cos_gwtheta": (-1.0, 1.0),
    "gwphi": (0.0, 2 * 3.141592653589793),
    "log10_fgw": (-9.0, -7.0),
    "cos_inc": (-1.0, 1.0),
    "psi": (0.0, 3.141592653589793),
    "phase0": (0.0, 2 * 3.141592653589793),
}
CW_PARAM_ORDER = list(CW_BOUNDS.keys())

In [ ]:
from __future__ import annotations

import warnings
from io import StringIO

import blackjax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import pint.models as pm
from loguru import logger
logger.disable("pint")

from jaxpint.fitters import WLSFitter
from jaxpint.likelihood import single_pulsar_logL
from jaxpint.pta.likelihood import PTAConfig, pta_logL
from jaxpint.pta.batching import BatchedPTAConfig, pta_logL_batched
from jaxpint.pta.params import GlobalParams
from jaxpint.pta.signals.cw import CWInjectorStack
from jaxpint.notebook_utils import (
    generate_random_par,
    inject_and_build_config,
    setup_synthetic_pta,
)

print(jax.devices())


## 1. Generate Synthetic Pulsars

Each pulsar gets a random sky position, spin frequency, spindown rate, DM, and
distance. We build a minimal `.par` string and parse it with PINT.

**Convention:** `CWInjector` reads the `PX` parameter directly as distance in kpc.

In [ ]:
rng = np.random.default_rng(SEED)

par_strings = [
    generate_random_par(
        idx, rng,
        start_mjd=START_MJD,
        include_dm=True,
        free_params=True,
        extra_params={"EQUAD tel gbt": "0.1"},
    )
    for idx in range(N_PULSARS)
]
pint_models = [pm.get_model(StringIO(p)) for p in par_strings]

print(f"Generated {N_PULSARS} pulsars")
for i, m in enumerate(pint_models):
    print(f"  Pulsar {i}: {m.PSR.value}  free_params={m.free_params}")


In [ ]:
synthetic = setup_synthetic_pta(
    pint_models,
    start_mjd=START_MJD, end_mjd=END_MJD,
    n_toas=N_TOAS, toa_error_s=TOA_ERROR, freq_mhz=FREQ,
)
toa_data_list = list(synthetic.toa_data_list)
pulsar_params_list = list(synthetic.pulsar_params_list)
timing_models = list(synthetic.timing_models)
noise_models = list(synthetic.noise_models)
base_toa_data_list = list(synthetic.toa_data_list)

for i, model in enumerate(pint_models):
    px_val = float(pulsar_params_list[i].param_value("PX"))
    f0 = float(pulsar_params_list[i].param_value("F0"))
    n_free = pulsar_params_list[i].n_free
    print(f"  Pulsar {i}: {model.PSR.value:>20s}  PX={px_val:.2f} kpc  F0={f0:.1f} Hz  n_free={n_free}")

print(f"\nAll {N_PULSARS} pulsars loaded.")


## 2. Set Up CW Sources and Inject Signals

Create M CW sources with random sky positions and GW frequencies, then inject
the CW timing delays into the TOAs. The NUTS sampler will try to recover these
injected parameters.

In [ ]:
import astropy.units as u

positions_np = []
for model in pint_models:
    ra_rad = model.RAJ.quantity.to(u.rad).value
    dec_rad = model.DECJ.quantity.to(u.rad).value
    positions_np.append(np.array([
        np.cos(dec_rad) * np.cos(ra_rad),
        np.cos(dec_rad) * np.sin(ra_rad),
        np.sin(dec_rad),
    ]))
positions = jnp.array(np.array(positions_np))

per_source_values = []
for m in range(M_CW_SOURCES):
    vals = {
        "log10_h": -13.0,
        "cos_gwtheta": float(rng.uniform(-1, 1)),
        "gwphi": float(rng.uniform(0, 2 * np.pi)),
        "log10_fgw": float(rng.uniform(-9, -7)),
    }
    per_source_values.append(vals)
    print(
        f"  CW source {m}: cos_gwtheta={vals['cos_gwtheta']:.3f}, "
        f"gwphi={vals['gwphi']:.3f}, log10_fgw={vals['log10_fgw']:.2f}"
    )

cw_injector = CWInjectorStack(
    positions, n_sources=M_CW_SOURCES, per_source_values=per_source_values,
)

# Register + inject signals into base TOAs; the returned `config` is the
# post-injection PTAConfig referenced later in the notebook.
true_global_params, injected_config = inject_and_build_config(synthetic, (cw_injector,))
print(f"\nGlobal params ({true_global_params.n_params}): {true_global_params.names}")


In [ ]:
# Signals already injected by inject_and_build_config; extract the resulting
# TOA data for downstream cells that expect it as a list.
injected_toa_data_list = list(injected_config.toa_data_list)
toa_data_list = injected_toa_data_list

for i, td in enumerate(injected_toa_data_list):
    delta_t = jnp.asarray(td.tdb_frac) - jnp.asarray(base_toa_data_list[i].tdb_frac)
    print(f"  Pulsar {i}: max |CW delay| = {float(jnp.max(jnp.abs(delta_t))):.2e} s")

print("\nCW signals injected into all TOAs.")


## 3. Per-Pulsar WLS Fits

Fit each pulsar individually to get starting points and covariance estimates
for the timing parameters. The WLS fit will absorb some of the slowly-varying
CW signal into the timing parameters — this is expected and the joint NUTS
sampler will disentangle them.

In [7]:
wls_results = []
wls_params_list = []
wls_covs = []

# Fit on signal-free TOAs to get clean starting points and covariances.
# The injected CW signal would bias timing params if fitted without a CW model.
for i in range(N_PULSARS):
    fitter = WLSFitter(
        timing_models[i], toa_data_list[i], pulsar_params_list[i],
        noise_model=noise_models[i],
    )
    _ = fitter.fit_toas(maxiter=1)  # JIT warmup
    result = fitter.fit_toas(maxiter=900)

    wls_results.append(result)
    wls_params_list.append(result.params)
    wls_covs.append(np.array(result.covariance_matrix))

    print(f"  Pulsar {i}: chi2_r={result.reduced_chi2:.4f}  "
          f"free={result.params.free_names()}")

print(f"\nAll {N_PULSARS} WLS fits complete.")

  Pulsar 0: chi2_r=1.0195  free=('PX', 'RAJ', 'DECJ', 'F0', 'F1', 'DM')
  Pulsar 1: chi2_r=0.4263  free=('PX', 'RAJ', 'DECJ', 'F0', 'F1', 'DM')
  Pulsar 2: chi2_r=0.0125  free=('PX', 'RAJ', 'DECJ', 'F0', 'F1', 'DM')
  Pulsar 3: chi2_r=4.9999  free=('PX', 'RAJ', 'DECJ', 'F0', 'F1', 'DM')
  Pulsar 4: chi2_r=0.0039  free=('PX', 'RAJ', 'DECJ', 'F0', 'F1', 'DM')

All 5 WLS fits complete.


## 4. Build Joint Parameter Space

The sampler operates in an unconstrained space where all parameters are O(1):

- **Timing params**: whitened via WLS Cholesky, `x_timing = mean + L @ theta_timing`
- **CW params**: logit-transformed, `x_cw = lo + (hi - lo) * sigmoid(theta_cw)`

This gives smooth gradients everywhere (no hard boundaries) and good conditioning.

In [8]:
# Build PTA config
config = PTAConfig(
    toa_data_list=tuple(toa_data_list),
    timing_models=tuple(timing_models),
    noise_models=tuple(noise_models),
    signal_injectors=(cw_injector,),
)

# Build batched config (single fused XLA trace)
batched_config = BatchedPTAConfig(
    toa_data_list=tuple(toa_data_list),
    timing_models=tuple(timing_models),
    noise_models=tuple(noise_models),
    signal_injectors=(cw_injector,),
    pulsar_params=tuple(wls_params_list),
)
# Dimensions
n_cw_per_source = len(CW_PARAM_ORDER)
n_cw_total = M_CW_SOURCES * n_cw_per_source
n_free_per_pulsar = [wls_params_list[i].n_free for i in range(N_PULSARS)]
n_timing_total = sum(n_free_per_pulsar)
n_total = n_cw_total + n_timing_total
print(f"Parameter space: {n_total} dimensions")
print(f"  CW: {M_CW_SOURCES} sources x {n_cw_per_source} params = {n_cw_total}")
print(f"  Timing: {N_PULSARS} pulsars x {n_free_per_pulsar} free = {n_timing_total}")

Parameter space: 44 dimensions
  CW: 2 sources x 7 params = 14
  Timing: 5 pulsars x [6, 6, 6, 6, 6] free = 30


/tmp/ipykernel_121616/1072653186.py:2: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  config = PTAConfig(
/home/hector/NYU/PTA/jax_pint/JaxPINT/jaxpint/pta/batching.py:160: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  self.config = PTAConfig(


In [9]:
# --- CW logit transform ---
# Build bounds arrays matching GlobalParams layout
cw_lo = jnp.array([CW_BOUNDS[k][0] for k in CW_PARAM_ORDER] * M_CW_SOURCES)
cw_hi = jnp.array([CW_BOUNDS[k][1] for k in CW_PARAM_ORDER] * M_CW_SOURCES)


def sigmoid(x):
    return jax.nn.sigmoid(x)


def cw_unconstrained_to_physical(theta_cw):
    """Map unconstrained theta_cw -> physical CW param values via sigmoid."""
    return cw_lo + (cw_hi - cw_lo) * sigmoid(theta_cw)


def cw_physical_to_unconstrained(cw_phys):
    """Map physical CW param values -> unconstrained theta_cw via logit."""
    t = (cw_phys - cw_lo) / (cw_hi - cw_lo)
    t = jnp.clip(t, 1e-7, 1.0 - 1e-7)
    return jnp.log(t / (1.0 - t))


def cw_log_jacobian(theta_cw):
    """Log |det J| of the sigmoid transform (sum of per-element log-Jacobians)."""
    s = sigmoid(theta_cw)
    return jnp.sum(jnp.log(s) + jnp.log1p(-s) + jnp.log(cw_hi - cw_lo))


# --- Per-pulsar timing whitening ---
# Each pulsar: x_free = wls_mean + L @ theta_timing_block
wls_means = [jnp.array(wls_params_list[i].free_values()) for i in range(N_PULSARS)]
wls_Ls = []
for i in range(N_PULSARS):
    cov = jnp.array(wls_covs[i])
    L = jnp.linalg.cholesky(cov)
    wls_Ls.append(L)


# --- Combined theta -> structured params ---
def theta_to_params(theta):
    """Map unconstrained theta -> (GlobalParams, tuple[ParameterVector]).

    Layout: theta = [theta_cw (n_cw_total) | theta_timing_0 | theta_timing_1 | ...]
    """
    theta_cw = theta[:n_cw_total]
    cw_physical = cw_unconstrained_to_physical(theta_cw)

    # Rebuild GlobalParams with physical CW values
    gp = GlobalParams(cw_physical, true_global_params.names, true_global_params._name_to_index)

    # Rebuild per-pulsar ParameterVectors with whitened timing params
    offset = n_cw_total
    pp_list = []
    for i in range(N_PULSARS):
        n_f = n_free_per_pulsar[i]
        theta_block = theta[offset : offset + n_f]
        free_vals = wls_means[i] + wls_Ls[i] @ theta_block
        pp_list.append(wls_params_list[i].with_free_values(free_vals))
        offset += n_f

    return gp, tuple(pp_list)


# --- True parameter values in theta-space (should be ~0 for timing, logit(true) for CW) ---
true_cw_physical = true_global_params.values

# --- Initialization: zeros = WLS center (timing) + prior midpoint (CW) ---
# In a real analysis we have no knowledge of the true CW parameters,
# so we start at the midpoint of each prior (theta_cw=0 maps to the center
# of each bounded interval via sigmoid). Timing params start at WLS solution.
theta_init = jnp.zeros(n_total)

print(f"theta_init shape: {theta_init.shape}")
print(f"CW init (physical): {cw_unconstrained_to_physical(theta_init[:n_cw_total])}")

theta_init shape: (44,)
CW init (physical): [-14.           0.           3.14159265  -8.           0.
   1.57079633   3.14159265 -14.           0.           3.14159265
  -8.           0.           1.57079633   3.14159265]


## 5. Log-Probability

The log-posterior in unconstrained (theta) space:

$$\log p(\theta) = \log L_{\rm PTA}(\theta) - \tfrac{1}{2}\theta_{\rm timing}^T \theta_{\rm timing} + \log|J_{\rm sigmoid}|$$

- Likelihood: `pta_logL` sums per-pulsar likelihoods with CW signal injection
- Timing prior: standard normal in whitened space (= Gaussian WLS prior)
- CW prior: uniform in physical space, with sigmoid Jacobian correction


In [10]:
def log_prob(theta):
    """Log-posterior in unconstrained theta-space."""
    theta_cw = theta[:n_cw_total]
    theta_timing = theta[n_cw_total:]

    # Transform to structured params
    gp, pp = theta_to_params(theta)

    # Likelihood
    ll = pta_logL_batched(gp, pp, batched_config)

    # Timing prior: standard normal in whitened space
    log_prior_timing = -0.5 * jnp.dot(theta_timing, theta_timing)

    # CW prior: uniform in physical space -> Jacobian correction
    log_jac = cw_log_jacobian(theta_cw)

    return ll + log_prior_timing + log_jac


# Test at true values
log_prob_jit = jax.jit(log_prob)
grad_log_prob_jit = jax.jit(jax.grad(log_prob))

# Initialize at middle of prior values

lp_val = log_prob_jit(theta_init)
grad_val = grad_log_prob_jit(theta_init)

print(f"log P at true values: {lp_val:.4f}")
print(f"Gradient norm: {jnp.linalg.norm(grad_val):.4f}")
print(f"Any NaN gradients: {jnp.any(jnp.isnan(grad_val))}")
print(f"Any Inf gradients: {jnp.any(jnp.isinf(grad_val))}")

log P at true values: 9557.7518
Gradient norm: 45678.1484
Any NaN gradients: False
Any Inf gradients: False


## 6. NUTS Warmup and Sampling

In [11]:
%%time
# Warmup with window adaptation
warmup = blackjax.window_adaptation(
    blackjax.nuts,
    log_prob,
    initial_step_size=0.5,
)

rng_key = jax.random.key(SEED)
rng_key, warmup_key = jax.random.split(rng_key)

print(f"Running {N_WARMUP}-step warmup ({n_total} dimensions)...")
print("(First call includes JIT compilation — may take a few minutes)")
(state, warmup_info), _ = warmup.run(warmup_key, theta_init, num_steps=N_WARMUP)

print(f"\nAdapted step size: {warmup_info['step_size']:.6f}")
print(f"Final state log-prob: {state.logdensity:.4f}")

Running 1000-step warmup (44 dimensions)...
(First call includes JIT compilation — may take a few minutes)

Adapted step size: 0.000254
Final state log-prob: 9589.8428
CPU times: user 1h 18min 54s, sys: 2.64 s, total: 1h 18min 56s
Wall time: 1h 18min 56s


In [ ]:
%%time
# Build the NUTS kernel with adapted parameters
nuts_kernel = blackjax.nuts(
    log_prob,
    step_size=warmup_info["step_size"],
    inverse_mass_matrix=warmup_info["inverse_mass_matrix"],
)


def inference_step(carry, rng_key):
    state, info = nuts_kernel.step(rng_key, carry)
    return state, (state, info)


# Run sampling
rng_key, sample_key = jax.random.split(rng_key)
sample_keys = jax.random.split(sample_key, N_SAMPLES)

print(f"Drawing {N_SAMPLES} samples...")
(final_state, (states, infos)) = jax.lax.scan(inference_step, state, sample_keys)

acceptance_rate = float(jnp.mean(infos.acceptance_rate))
print(f"Acceptance rate: {acceptance_rate:.2%}")
print(f"Samples shape: {states.position.shape}")

Drawing 2000 samples...


## 7. Diagnostics

In [ ]:
# Transform all samples to physical space
theta_samples = states.position  # (N_SAMPLES, n_total)

# CW samples: transform from unconstrained to physical
cw_samples_unconstrained = theta_samples[:, :n_cw_total]
cw_samples_physical = jax.vmap(cw_unconstrained_to_physical)(cw_samples_unconstrained)
cw_samples_np = np.array(cw_samples_physical)

# Per-pulsar timing samples: transform from whitened to physical
timing_samples_per_pulsar = []
offset = n_cw_total
for i in range(N_PULSARS):
    n_f = n_free_per_pulsar[i]
    theta_blocks = theta_samples[:, offset : offset + n_f]
    # x_free = wls_mean + L @ theta_block (vectorized over samples)
    physical = jax.vmap(lambda tb, m=wls_means[i], L=wls_Ls[i]: m + L @ tb)(theta_blocks)
    timing_samples_per_pulsar.append(np.array(physical))
    offset += n_f

print(f"CW samples: {cw_samples_np.shape}")
print(f"Timing samples per pulsar: {[s.shape for s in timing_samples_per_pulsar]}")

In [ ]:
# NUTS efficiency metrics
print(f"Acceptance rate: {acceptance_rate:.2%}")
print(f"Mean tree depth: {np.mean(np.array(infos.num_integration_steps)):.1f}")
print(f"Max tree depth: {np.max(np.array(infos.num_integration_steps))}")
print(f"Divergent transitions: {np.sum(np.array(infos.is_divergent))}")

### CW Parameter Trace Plots

Each row shows one CW parameter across all sources. Red dashed lines mark the
injected true values.

In [ ]:
true_cw_np = np.array(true_global_params.values)

fig, axes = plt.subplots(n_cw_total, 2, figsize=(14, 2.5 * n_cw_total))
if n_cw_total == 1:
    axes = axes[np.newaxis, :]

for k in range(n_cw_total):
    # Determine source index and param name
    src_idx = k // n_cw_per_source
    param_idx = k % n_cw_per_source
    param_name = f"cw{src_idx}_{CW_PARAM_ORDER[param_idx]}"

    # Trace plot
    axes[k, 0].plot(cw_samples_np[:, k], alpha=0.7, linewidth=0.5)
    axes[k, 0].axhline(true_cw_np[k], color="r", linestyle="--", label="True")
    axes[k, 0].set_ylabel(param_name, fontsize=8)
    axes[k, 0].legend(loc="upper right", fontsize=7)

    # Histogram
    axes[k, 1].hist(cw_samples_np[:, k], bins=50, density=True, alpha=0.7)
    axes[k, 1].axvline(true_cw_np[k], color="r", linestyle="--", label="True")
    axes[k, 1].legend(loc="upper right", fontsize=7)

axes[0, 0].set_title("Trace")
axes[0, 1].set_title("Marginal posterior")
axes[-1, 0].set_xlabel("Sample")
axes[-1, 1].set_xlabel("Value")
plt.tight_layout()
plt.show()

### CW Corner Plots

Pairwise scatter plots for each CW source's 7 parameters.

In [ ]:
for src in range(M_CW_SOURCES):
    src_start = src * n_cw_per_source
    src_end = src_start + n_cw_per_source
    src_samples = cw_samples_np[:, src_start:src_end]
    src_true = true_cw_np[src_start:src_end]
    labels = CW_PARAM_ORDER

    fig, axes = plt.subplots(n_cw_per_source, n_cw_per_source,
                             figsize=(14, 14))
    for i in range(n_cw_per_source):
        for j in range(n_cw_per_source):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False)
            elif i == j:
                ax.hist(src_samples[:, i], bins=40, density=True, alpha=0.7)
                ax.axvline(src_true[i], color="r", linestyle="--")
            else:
                ax.scatter(src_samples[:, j], src_samples[:, i],
                           s=1, alpha=0.3)
                ax.axhline(src_true[i], color="r", linestyle="--", lw=0.5)
                ax.axvline(src_true[j], color="r", linestyle="--", lw=0.5)

            if i == n_cw_per_source - 1:
                ax.set_xlabel(labels[j], fontsize=7)
            if j == 0:
                ax.set_ylabel(labels[i], fontsize=7)
            ax.tick_params(labelsize=5)

    fig.suptitle(f"CW source {src}", fontsize=14)
    plt.tight_layout()
    plt.show()

### CW Parameter Recovery Summary

In [ ]:
print(f"{'Parameter':<25} {'True':>12} {'Post Mean':>12} {'Post Std':>12} {'Diff (σ)':>10}")
print("-" * 72)

for src in range(M_CW_SOURCES):
    for k, pname in enumerate(CW_PARAM_ORDER):
        idx = src * n_cw_per_source + k
        true_val = true_cw_np[idx]
        post_mean = np.mean(cw_samples_np[:, idx])
        post_std = np.std(cw_samples_np[:, idx])
        diff_sigma = (post_mean - true_val) / post_std if post_std > 0 else 0.0
        label = f"cw{src}_{pname}"
        print(f"{label:<25} {true_val:>12.4f} {post_mean:>12.4f} {post_std:>12.4g} {diff_sigma:>10.2f}")
    print()

### Per-Pulsar Timing Parameter Recovery

In [ ]:
for i in range(N_PULSARS):
    samples_i = timing_samples_per_pulsar[i]
    free_names = wls_params_list[i].free_names()
    true_free = np.array(pulsar_params_list[i].free_values())
    wls_free = np.array(wls_params_list[i].free_values())
    wls_unc = np.array(wls_results[i].parameter_uncertainties)

    print(f"\nPulsar {i}: {pint_models[i].PSR.value}")
    print(f"  {'Param':<10} {'True':>18} {'WLS':>18} {'Post Mean':>18} {'Post Std':>12} {'Δ(σ)':>8}")
    print("  " + "-" * 80)
    for j, name in enumerate(free_names):
        post_mean = np.mean(samples_i[:, j])
        post_std = np.std(samples_i[:, j])
        diff = (post_mean - true_free[j]) / post_std if post_std > 0 else 0.0
        print(f"  {name:<10} {true_free[j]:>18.10g} {wls_free[j]:>18.10g} "
              f"{post_mean:>18.10g} {post_std:>12.4g} {diff:>8.2f}")

### Effective Sample Size

ESS estimates the number of independent samples by accounting for autocorrelation.
Parameters with ESS < 100 may need longer chains.

In [ ]:
def ess_bulk(chain):
    """Estimate bulk ESS for a 1-D chain using initial positive sequence estimator."""
    n = len(chain)
    chain = chain - np.mean(chain)
    # Compute autocorrelation via FFT
    fft_chain = np.fft.fft(chain, n=2 * n)
    acf = np.fft.ifft(fft_chain * np.conj(fft_chain))[:n].real
    acf /= acf[0]
    # Sum consecutive pairs until negative
    tau = 1.0
    for t in range(1, n // 2):
        pair_sum = acf[2 * t - 1] + acf[2 * t]
        if pair_sum < 0:
            break
        tau += 2.0 * pair_sum
    return n / tau


# Compute ESS for all parameters
all_theta_samples = np.array(theta_samples)
all_names = []
for src in range(M_CW_SOURCES):
    for pname in CW_PARAM_ORDER:
        all_names.append(f"cw{src}_{pname}")
for i in range(N_PULSARS):
    for name in wls_params_list[i].free_names():
        all_names.append(f"psr{i}_{name}")

ess_values = [ess_bulk(all_theta_samples[:, k]) for k in range(n_total)]

print(f"{'Parameter':<25} {'ESS':>8}")
print("-" * 35)
for name, ess in zip(all_names, ess_values):
    flag = " <<<" if ess < 100 else ""
    print(f"{name:<25} {ess:>8.0f}{flag}")

print(f"\nMin ESS: {min(ess_values):.0f}")
print(f"Median ESS: {np.median(ess_values):.0f}")